# B3MN — Marigold surface normals as a multiplicity cue

## Why
Geometry has helped only as multiplicity inside merged regions, and **depth edges (K3)** were the strongest form (GPV_K held-out S6 278 vs 324). In top-down scenes, adjacent objects often sit at almost the same depth, so depth edges are weak — but each object's curved surface turns away at its boundary. **Surface normals** may show boundaries that depth misses.

## Pre-declared design (before any B3MN result)
| Choice | Rule |
|---|---|
| Normals | `prs-eth/marigold-normals-v1-1` (diffusers `MarigoldNormalsPipeline`), fp16, 4 steps, ensemble 1, processing resolution 768, seed 0 — the same settings as our depth model |
| Edge map | per pixel: max over its 4 neighbours of (1 − cosine between unit normals); Gaussian-smoothed, σ 1 |
| K3n | K3's watershed unchanged (h-minima, h = 0.1 × (p95 − p5) of the region's edge values; segments ≥ 0.25 × A_ref; ≥ 1 per region), with the **normal-edge map** as elevation |
| GPV_K-n | GPV_K_v1 with K3n replacing K3 (rule A, regions, Gate 1 unchanged) |
| H1b-K3n | B3U's H1b-K3 definition with K3n (H1b's SG gate decides; anchor + K3n over qualifying merged regions) |
| Compared with | GPV_K_v1, H1b-K3 (depth edges, B3U), H1b_v1 |
| Targets | T1, T2′, T3′ |

## Run order
T4 → 1.0–1.2 → 2.1 (install) → 2.2 (normals caching, ~10 min, resumable) → 3.1 (CPU-capable) → 3.2.

# 1. Setup

In [11]:
# 1.0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# ============================================================
# Step 1.1 — Paths, repo modules, train pool, frozen-system counts
# ============================================================

import importlib, json, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
FSC147_RAW = PROJECT_ROOT / "datasets" / "FSC147" / "raw"
FSC147_IMAGE_DIR = FSC147_RAW / "images_384_VarV2"
TD_ROOT = PROJECT_ROOT / "cache" / "FSC147_train_dev400"
B3S_DIR = RESULTS_ROOT / "B3S_Semantic_Improvement"
OUT_DIR = RESULTS_ROOT / "B3MN_Marigold_Normals"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

import src.pipeline.io_cache as io_cache
import src.pipeline.sg as sg
import src.pipeline.gpv as gpv
import src.pipeline.hybrid as hybrid
import src.pipeline.b0 as b0
for m in (io_cache, sg, gpv, hybrid, b0):
    importlib.reload(m)

STRATA = ["S1_0_20", "S2_20_40", "S3_40_60", "S4_60_80", "S5_80_95", "S6_95_100"]
W = {"S1_0_20": .20, "S2_20_40": .20, "S3_40_60": .20, "S4_60_80": .20, "S5_80_95": .15, "S6_95_100": .05}
TD_DIRS = {"b0_masks": TD_ROOT / "b0_masks", "a3c_masks": TD_ROOT / "a3c_masks",
           "depth": TD_ROOT / "depth", "m1_split_labels": TD_ROOT / "m1_split_labels"}
SG_CFG = sg.SGConfig.from_frozen_json(RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "SG_v1.1_frozen_config.json")

td_df = pd.read_csv(TD_ROOT / "FSC147_train_dev400.csv")
td_res = pd.read_csv(B3S_DIR / "B3S_step1_per_image.csv")
assert len(td_df) == 400 and len(td_res) == 400
print("Repo commit:", REPO_COMMIT, "| output:", OUT_DIR)

Repo commit: d70bb1d6d631c12156ba7bf1881d22f5b2eb6100 | output: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3MN_Marigold_Normals


In [13]:
import torch

# ============================================================
# Step 1.2 — Shared helpers: guided masks, B0 meta filtering, systems, targets
# ============================================================

def guided_masks_for_prompts(predictor, image_np, prompts_xy):
    h, w = image_np.shape[:2]
    masks, scores = [], []
    if len(prompts_xy):
        with torch.inference_mode():
            predictor.set_image(image_np)
            for p in prompts_xy:
                m, s, _ = predictor.predict(point_coords=np.asarray([p], dtype=np.float32),
                                            point_labels=np.asarray([1], dtype=np.int32),
                                            multimask_output=False)
                masks.append(np.asarray(m[0], dtype=bool)); scores.append(float(np.asarray(s)[0]))
    return (np.asarray(masks) if masks else np.zeros((0, h, w), bool)), np.asarray(scores, float)

def filter_meta(meta, keep):
    return {k: np.asarray(v)[keep] for k, v in meta.items()}

def systems_from_masks(b0_masks, b0_iou, g_masks, g_scores, a3c, depth, san_target):
    keep = gpv.nms_masks(g_masks, g_scores, 0.5) if g_masks.shape[0] else np.zeros(0, int)
    r4_m, _, _ = gpv.build_r4(g_masks[keep], np.asarray(g_scores)[keep], b0_masks, b0_iou, 0.5)
    d1 = gpv.d1_peaks(san_target)
    h1b = float(hybrid.h1b_count(r4_m, d1, a3c, depth, SG_CFG)[0])
    return {"B0": int(b0_masks.shape[0]), "GPV_R4": int(r4_m.shape[0]), "H1b": h1b}

def evaluate(frame, systems, label):
    gt = frame["gt_count"].to_numpy(float); st = frame["density_stratum"].to_numpy()
    AE = {s: np.abs(frame[s].to_numpy(float) - gt) for s in systems}
    IDX = {}
    for j, k in enumerate(STRATA):
        pos = np.flatnonzero(st == k)
        IDX[k] = pos[np.random.default_rng(1000 + j).integers(0, len(pos), size=(10_000, len(pos)))]
    def sci(a, b, k):
        d = AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)
        return AE[a][st == k].mean() - AE[b][st == k].mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)
    def rci(a, b):
        d = sum(w * (AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)) for k, w in W.items())
        o = sum(w * (AE[a][st == k].mean() - AE[b][st == k].mean()) for k, w in W.items())
        return o, np.percentile(d, 2.5), np.percentile(d, 97.5)
    res = pd.DataFrame([{"system": s,
                         "MAE_reweighted": sum(w * AE[s][st == k].mean() for k, w in W.items()),
                         "mean_signed": float((frame[s] - frame["gt_count"]).mean()),
                         **{f"MAE_{k[:2]}": AE[s][st == k].mean() for k in STRATA}} for s in systems])
    rows = []
    for s in [x for x in systems if x not in ("B0_v1", "GPV_R4_v1", "H1b_v1")]:
        rec = {"system": s}; t1 = []; t2 = []
        for k in ["S1_0_20", "S2_20_40"]:
            d, lo, hi = sci(s, "B0_v1", k); rec[f"{k[:2]} vs B0"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t1.append(lo <= 0)
        for k in ["S5_80_95", "S6_95_100"]:
            d, lo, hi = sci(s, "H1b_v1", k); rec[f"{k[:2]} vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t2.append(lo <= 0)
        d, lo, hi = rci(s, "H1b_v1"); rec["overall vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"
        rec.update({"T1": all(t1), "T2'": all(t2), "T3'": hi < 0, "ALL": all(t1) and all(t2) and hi < 0})
        rows.append(rec)
    print(f"{label} — metrics:"); display(res.round(2))
    print("Targets (difference [95% CI]; negative = system better):"); display(pd.DataFrame(rows))
    return res, pd.DataFrame(rows)
print("helpers ready")

helpers ready


# 2. Marigold normals

In [ ]:
# ============================================================
# Step 2.1 — Install diffusers (same versions as M1)
# ============================================================

!pip install -q "accelerate==1.14.0" "diffusers==0.40.0" "transformers==5.16.1"
import importlib.metadata as md_meta
print({k: md_meta.version(k) for k in ["diffusers", "transformers", "accelerate"]})

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 88.9 MB/s eta 0:00:00
{'diffusers': '0.40.0', 'transformers': '5.16.1', 'accelerate': '1.14.0'}


In [ ]:
# ============================================================
# Step 2.2 — Normals for train_dev400 (GPU, resumable)
# ============================================================

import torch
from PIL import Image
from diffusers import MarigoldNormalsPipeline
assert torch.cuda.is_available(), "Switch the runtime to T4."
DEVICE = "cuda"
NORM_ID = "prs-eth/marigold-normals-v1-1"
pipe_n = MarigoldNormalsPipeline.from_pretrained(NORM_ID, dtype=torch.float16).to(DEVICE)
pipe_n.set_progress_bar_config(disable=True)

N_ROOT = TD_ROOT / "normals"
N_ROOT.mkdir(parents=True, exist_ok=True)
todo = [r for r in td_df.itertuples(index=False)
        if not (N_ROOT / f"{io_cache.image_stem(r.image_id)}.npz").exists()]
print(f"Done: {len(td_df) - len(todo)} | to process: {len(todo)}")
t0 = time.time()
for r in tqdm(todo, desc="Normals"):
    img = Image.open(FSC147_IMAGE_DIR / r.image_id).convert("RGB")
    g = torch.Generator(device=DEVICE).manual_seed(0)
    with torch.inference_mode():
        out = pipe_n(img, num_inference_steps=4, ensemble_size=1, processing_resolution=768, generator=g)
    nrm = np.squeeze(np.asarray(out.prediction)).astype(np.float32)      # (H, W, 3) in [-1, 1]
    assert nrm.shape[:2] == (img.size[1], img.size[0]) and nrm.shape[2] == 3, nrm.shape
    np.savez_compressed(N_ROOT / f"{io_cache.image_stem(r.image_id)}.npz", normals=nrm.astype(np.float16))
print(f"Elapsed: {(time.time() - t0) / 60:.1f} min")

model_index.json:   0%|          | 0.00/512 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

Done: 0 | to process: 400


Normals:   0%|          | 0/400 [00:00<?, ?it/s]

Elapsed: 11.5 min


# 3. K3n inside GPV_K and H1b

In [ ]:
# ============================================================
# Step 3.1 — GPV_K-n and H1b-K3n (CPU-capable)
# ============================================================

from scipy import ndimage
import src.pipeline.multiplicity as K
importlib.reload(K)
N_ROOT = TD_ROOT / "normals"

def normal_edge_map(nrm):
    n = nrm.astype(np.float32)
    n = n / np.maximum(np.linalg.norm(n, axis=2, keepdims=True), 1e-6)
    edge = np.zeros(n.shape[:2], np.float32)
    for dy, dx in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
        sh = np.roll(n, shift=(dy, dx), axis=(0, 1))
        edge = np.maximum(edge, 1.0 - (n * sh).sum(axis=2))
    edge[0, :] = edge[1, :]; edge[-1, :] = edge[-2, :]; edge[:, 0] = edge[:, 1]; edge[:, -1] = edge[:, -2]
    return ndimage.gaussian_filter(edge, 1.0)

def k3n_segments(region, edge, min_seg, h_frac=0.1):
    e = edge[region]
    h = h_frac * float(np.percentile(e, 95) - np.percentile(e, 5))
    if h <= 0:
        return 1
    return K._marker_watershed(edge, region, -edge, h, min_seg)

def gpv_k_n_count(r4_m, r4_s, d1, edge):
    # GPV_K_v1 logic (rule A -> regions -> Gate 1) with K3n
    masks = np.asarray(r4_m, bool)
    if masks.shape[0] == 0:
        return 0
    peaks = gpv.points_per_mask(masks, d1)
    flagged = gpv.merged_mask_flags(masks, peaks, "A")
    a_ref = K.reference_area(masks, flagged)
    n_nonflag = int((~flagged).sum())
    regions = [rg for _, rg in K.build_regions(masks, r4_s, flagged) if rg.sum() > 0 and rg.sum() >= 0.5 * a_ref]
    if not (regions and sum(K.k1_peaks(rg, d1) for rg in regions) > n_nonflag):
        return int(masks.shape[0])
    return n_nonflag + sum(max(1, k3n_segments(rg, edge, 0.25 * a_ref)) for rg in regions)

def h1b_k3n_count(r4_m, d1, a3c, depth, edge):
    # B3U H1b-K3 definition with K3n
    base, info = hybrid.h1b_count(r4_m, d1, a3c, depth, SG_CFG)
    if not info["stage2_on"] or r4_m.shape[0] == 0:
        return base
    merged = gpv.merged_mask_flags(r4_m, gpv.points_per_mask(r4_m, d1), "A")
    if not merged.any():
        return base
    anchor = r4_m[~merged]
    areas = anchor.reshape(anchor.shape[0], -1).sum(1) if anchor.shape[0] else np.zeros(0)
    a_ref = float(np.median(areas)) if areas.size else float(np.median(r4_m[merged].reshape(merged.sum(), -1).sum(1)))
    taken = anchor.any(axis=0) if anchor.shape[0] else np.zeros(r4_m.shape[1:], bool)
    total = int(anchor.shape[0])
    for i in np.flatnonzero(merged):
        rg = r4_m[i] & ~taken
        taken |= rg
        if rg.sum() > 0 and rg.sum() >= 0.5 * a_ref:
            total += max(1, k3n_segments(rg, edge, 0.25 * a_ref))
    return float(total)

rows = []
for r in tqdm(td_df.itertuples(index=False), total=len(td_df), desc="K3n systems"):
    di, stem = r.dataset_index, io_cache.image_stem(r.image_id)
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    b0m, b0meta = io_cache.load_b0_masks(p["b0_masks"])
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    dep = io_cache.load_depth(p["depth"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    g = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery" / f"{int(di):04d}_{stem}.npz")
    with np.load(N_ROOT / f"{stem}.npz") as v:
        edge = normal_edge_map(v["normals"])
    kg = gpv.nms_masks(g["masks"], g["scores"], 0.5)
    r4_m, r4_s, _ = gpv.build_r4(g["masks"][kg], g["scores"][kg], b0m, b0meta["predicted_iou"], 0.5)
    d1 = gpv.d1_peaks(san["target_score"])
    rows.append({"dataset_index": di, "GPV_K-n": gpv_k_n_count(r4_m, r4_s, d1, edge),
                 "H1b-K3n": h1b_k3n_count(r4_m, d1, a3c, dep, edge)})
kn = pd.DataFrame(rows)

K3n systems:   0%|          | 0/400 [00:00<?, ?it/s]

In [ ]:
# ============================================================
# Step 3.2 — Metrics and targets (normals vs depth edges)
# ============================================================

b3u = pd.read_csv(RESULTS_ROOT / "B3U_Unified_Multiplicity" / "B3U_per_image.csv")[["dataset_index", "H1b-K3"]]
frame = (kn.merge(b3u, on="dataset_index")
         .merge(td_res[["dataset_index", "gt_count", "density_stratum", "B0", "GPV_R4_v1", "GPV_K_v1", "H1b_v1"]]
                .rename(columns={"B0": "B0_v1"}), on="dataset_index", validate="one_to_one"))
SYS = ["B0_v1", "GPV_R4_v1", "H1b_v1", "GPV_K_v1", "H1b-K3", "GPV_K-n", "H1b-K3n"]
res, targets = evaluate(frame, SYS, "Marigold normals (K3n) vs depth edges (K3) — train_dev400")

frame.to_csv(OUT_DIR / "B3MN_per_image.csv", index=False)
res.to_csv(OUT_DIR / "B3MN_metrics.csv", index=False)
targets.to_csv(OUT_DIR / "B3MN_targets.csv", index=False)
with open(OUT_DIR / "B3MN_metadata.json", "w") as f:
    json.dump({"notebook": "B3MN_Marigold_Normals",
               "normals": "prs-eth/marigold-normals-v1-1, fp16, 4 steps, ensemble 1, 768 px, seed 0",
               "edge_map": "max over 4-neighbours of (1 - cos) between unit normals; Gaussian sigma 1",
               "K3n": "K3 watershed (h-minima h=0.1*(p95-p5), segments >= 0.25 A_ref) on the normal-edge map",
               "variants": {"GPV_K-n": "GPV_K_v1 with K3n", "H1b-K3n": "B3U H1b-K3 definition with K3n"},
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", OUT_DIR)

Marigold normals (K3n) vs depth edges (K3) — train_dev400 — metrics:


,system,MAE_reweighted,mean_signed,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0_v1,35.67,-62.67,5.56,7.74,13.30,22.93,59.03,338.08
1,GPV_R4_v1,29.94,-46.78,8.62,7.70,12.45,16.13,40.52,297.63
2,H1b_v1,28.65,-41.28,10.46,9.49,12.35,14.20,37.33,274.98
3,GPV_K_v1,32.35,-20.68,14.90,11.08,26.87,15.22,36.82,264.38
4,H1b-K3,37.32,-18.66,10.01,10.38,16.83,39.67,51.95,282.90
5,GPV_K-n,27.89,-36.12,11.60,8.34,13.70,15.15,35.53,256.10
6,H1b-K3n,28.41,-36.35,10.41,8.68,14.87,15.17,35.70,264.70


Targets (difference [95% CI]; negative = system better):


,system,S1 vs B0,S2 vs B0,S5 vs H1b,S6 vs H1b,overall vs H1b,T1,T2',T3',ALL
0,GPV_K_v1,"+9.34 [+5.00, +14.48]","+3.34 [+0.44, +6.83]","-0.52 [-6.67, +6.37]","-10.60 [-42.03, +20.08]","+3.70 [-0.56, +10.16]",False,True,False,False
1,H1b-K3,"+4.45 [+2.75, +6.20]","+2.64 [-0.03, +5.89]","+14.62 [-0.82, +38.35]","+7.92 [-14.93, +34.02]","+8.67 [+1.49, +19.46]",False,True,False,False
2,GPV_K-n,"+6.04 [+3.49, +8.88]","+0.60 [-1.12, +2.47]","-1.80 [-8.45, +5.32]","-18.88 [-42.75, +2.25]","-0.76 [-2.62, +1.11]",False,True,False,False
3,H1b-K3n,"+4.85 [+2.89, +6.96]","+0.94 [-0.80, +2.81]","-1.63 [-4.22, +0.63]","-10.28 [-28.15, +6.57]","-0.24 [-1.50, +1.16]",False,True,False,False


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3MN_Marigold_Normals


## Section 3 result

**Surface normals are a far better multiplicity cue than depth edges** under identical rules: GPV_K-n reweighted **27.89** (H1b_v1 28.65; GPV_K_v1 32.35) — lowest of any system so far, n.s.; S5 35.53 and S6 256.1. H1b-K3n 28.41 (H1b-K3 37.32). Normals remove most of depth edges' false splitting (GPV_K S3 26.87 → 13.70; H1b-K3 S4 39.67 → 15.17) while keeping dense gains. T2′ passes for both; T1 fails (the base pipelines' sparse surplus, unchanged). Ordering: depth values (G2) < depth edges (K3) < surface-normal edges (K3n).

# 4. Normals multiplicity on other mask bases (LM0, and zoomed bases when ready)

## Pre-declared (before any Section 4 result)
- **GPV_K-n[base]:** GPV_K_v1 logic (rule A → regions → Gate 1 → K3n) on the base's GPV_R4 mask set.
- **H1b-K3n[base]:** B3U's H1b-K3 definition with K3n, on the base's GPV_R4 mask set.
- Bases evaluated = every base whose masks are complete for all 400 images:
  - **LM0** — SAM2.1-Large, masked input M0 (B3L §5 cache);
  - **Z-S** — SAM2.1-Small + zoom (B3Z [S] cache), when complete;
  - **Z-LM0** — SAM2.1-Large + masked + zoom (B3Z [LM0] cache), when complete.
- K3n, A3c, depth, D1 peaks and counting otherwise unchanged; targets T1, T2′, T3′.

Re-run this section after B3Z finishes to add the zoomed bases. CPU is enough (Section 1 first).

In [14]:
# ============================================================
# Step 4.1 — GPV_K-n and H1b-K3n on every complete mask base (CPU)
# ============================================================

from scipy import ndimage
import src.pipeline.multiplicity as K
importlib.reload(K)
N_ROOT = TD_ROOT / "normals"
BASE_DIRS = {"LM0": TD_ROOT / "sam2_large_masked_M0",
             "Z-S": TD_ROOT / "zoomed" / "S",
             "Z-LM0": TD_ROOT / "zoomed" / "LM0"}
ready = [b for b, d in BASE_DIRS.items() if (d / "_done").exists() and len(list((d / "_done").glob("*.json"))) == 400]
print("Complete bases:", ready)

# --- helpers from Section 3 (redefined so Section 4 can run after Section 1 only)
def normal_edge_map(nrm):
    n = nrm.astype(np.float32)
    n = n / np.maximum(np.linalg.norm(n, axis=2, keepdims=True), 1e-6)
    edge = np.zeros(n.shape[:2], np.float32)
    for dy, dx in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
        edge = np.maximum(edge, 1.0 - (n * np.roll(n, shift=(dy, dx), axis=(0, 1))).sum(axis=2))
    edge[0, :] = edge[1, :]; edge[-1, :] = edge[-2, :]; edge[:, 0] = edge[:, 1]; edge[:, -1] = edge[:, -2]
    return ndimage.gaussian_filter(edge, 1.0)

def k3n_segments(region, edge, min_seg, h_frac=0.1):
    e = edge[region]
    h = h_frac * float(np.percentile(e, 95) - np.percentile(e, 5))
    return 1 if h <= 0 else K._marker_watershed(edge, region, -edge, h, min_seg)

def gpv_k_n_count(r4_m, r4_s, d1, edge):
    masks = np.asarray(r4_m, bool)
    if masks.shape[0] == 0:
        return 0
    flagged = gpv.merged_mask_flags(masks, gpv.points_per_mask(masks, d1), "A")
    a_ref = K.reference_area(masks, flagged)
    n_nonflag = int((~flagged).sum())
    regions = [rg for _, rg in K.build_regions(masks, r4_s, flagged) if rg.sum() > 0 and rg.sum() >= 0.5 * a_ref]
    if not (regions and sum(K.k1_peaks(rg, d1) for rg in regions) > n_nonflag):
        return int(masks.shape[0])
    return n_nonflag + sum(max(1, k3n_segments(rg, edge, 0.25 * a_ref)) for rg in regions)

def h1b_k3n_count(r4_m, d1, a3c, depth, edge):
    base, info = hybrid.h1b_count(r4_m, d1, a3c, depth, SG_CFG)
    if not info["stage2_on"] or r4_m.shape[0] == 0:
        return base
    merged = gpv.merged_mask_flags(r4_m, gpv.points_per_mask(r4_m, d1), "A")
    if not merged.any():
        return base
    anchor = r4_m[~merged]
    areas = anchor.reshape(anchor.shape[0], -1).sum(1) if anchor.shape[0] else np.zeros(0)
    a_ref = float(np.median(areas)) if areas.size else float(np.median(r4_m[merged].reshape(merged.sum(), -1).sum(1)))
    taken = anchor.any(axis=0) if anchor.shape[0] else np.zeros(r4_m.shape[1:], bool)
    total = int(anchor.shape[0])
    for i in np.flatnonzero(merged):
        rg = r4_m[i] & ~taken
        taken |= rg
        if rg.sum() > 0 and rg.sum() >= 0.5 * a_ref:
            total += max(1, k3n_segments(rg, edge, 0.25 * a_ref))
    return float(total)

rows = []
for r in tqdm(td_df.itertuples(index=False), total=len(td_df), desc="K3n on bases"):
    di, stem = r.dataset_index, io_cache.image_stem(r.image_id)
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    dep = io_cache.load_depth(p["depth"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    d1 = gpv.d1_peaks(san["target_score"])
    with np.load(N_ROOT / f"{stem}.npz") as v:
        edge = normal_edge_map(v["normals"])
    row = {"dataset_index": di}
    for b in ready:
        d = BASE_DIRS[b]
        bm, bmeta = io_cache.load_b0_masks(d / "b0" / f"{int(di):05d}_{stem}.npz")
        g = io_cache.load_guided_masks(d / "guided" / f"{int(di):04d}_{stem}.npz")
        kg = gpv.nms_masks(g["masks"], g["scores"], 0.5) if g["masks"].shape[0] else np.zeros(0, int)
        r4_m, r4_s, _ = gpv.build_r4(g["masks"][kg], np.asarray(g["scores"])[kg], bm, bmeta["predicted_iou"], 0.5)
        row.update({f"GPV_R4[{b}]": int(r4_m.shape[0]),
                    f"H1b[{b}]": float(hybrid.h1b_count(r4_m, d1, a3c, dep, SG_CFG)[0]),
                    f"GPV_K-n[{b}]": gpv_k_n_count(r4_m, r4_s, d1, edge),
                    f"H1b-K3n[{b}]": h1b_k3n_count(r4_m, d1, a3c, dep, edge)})
    rows.append(row)
kb = pd.DataFrame(rows)

Complete bases: ['LM0', 'Z-S', 'Z-LM0']


K3n on bases:   0%|          | 0/400 [00:00<?, ?it/s]

In [15]:
# ============================================================
# Step 4.2 — Metrics and targets across bases
# ============================================================

prev = pd.read_csv(OUT_DIR / "B3MN_per_image.csv")[["dataset_index", "GPV_K-n", "H1b-K3n"]] \
    if (OUT_DIR / "B3MN_per_image.csv").exists() else None
frame4 = kb.merge(td_res[["dataset_index", "gt_count", "density_stratum", "B0", "GPV_R4_v1", "GPV_K_v1", "H1b_v1"]]
                  .rename(columns={"B0": "B0_v1"}), on="dataset_index", validate="one_to_one")
if prev is not None:
    frame4 = frame4.merge(prev.rename(columns={"GPV_K-n": "GPV_K-n[S]", "H1b-K3n": "H1b-K3n[S]"}), on="dataset_index")

SYS4 = ["B0_v1", "GPV_R4_v1", "H1b_v1", "GPV_K_v1"] + \
       [c for c in ["GPV_K-n[S]", "H1b-K3n[S]"] if c in frame4.columns] + \
       [f"{k}[{b}]" for b in ready for k in ["H1b", "GPV_K-n", "H1b-K3n"]]
res4, targets4 = evaluate(frame4, SYS4, "Normals multiplicity on mask bases (train_dev400)")

D4 = OUT_DIR / "section4_bases"
D4.mkdir(parents=True, exist_ok=True)
frame4.to_csv(D4 / "B3MN_s4_per_image.csv", index=False)
res4.to_csv(D4 / "B3MN_s4_metrics.csv", index=False)
targets4.to_csv(D4 / "B3MN_s4_targets.csv", index=False)
with open(D4 / "B3MN_s4_metadata.json", "w") as f:
    json.dump({"section": "4 — GPV_K-n / H1b-K3n on other mask bases",
               "bases_evaluated": ready,
               "bases": {"LM0": "SAM2.1-Large, masked input M0 (B3L s5)", "Z-S": "Small + zoom (B3Z S)",
                         "Z-LM0": "Large + masked + zoom (B3Z LM0)"},
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", D4)

Normals multiplicity on mask bases (train_dev400) — metrics:


,system,MAE_reweighted,mean_signed,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0_v1,35.67,-62.67,5.56,7.74,13.30,22.93,59.03,338.08
1,GPV_R4_v1,29.94,-46.78,8.62,7.70,12.45,16.13,40.52,297.63
2,H1b_v1,28.65,-41.28,10.46,9.49,12.35,14.20,37.33,274.98
3,GPV_K_v1,32.35,-20.68,14.90,11.08,26.87,15.22,36.82,264.38
4,GPV_K-n[S],27.89,-36.12,11.60,8.34,13.70,15.15,35.53,256.10
5,H1b-K3n[S],28.41,-36.35,10.41,8.68,14.87,15.17,35.70,264.70
6,H1b[LM0],28.91,-49.22,6.89,6.14,9.93,14.90,42.42,299.43
7,GPV_K-n[LM0],27.71,-41.95,8.31,5.62,10.73,15.57,39.80,273.78
8,H1b-K3n[LM0],28.24,-44.33,6.92,6.24,11.62,15.55,38.97,286.55
9,H1b[Z-S],29.73,-24.72,16.30,14.45,18.70,20.08,32.45,219.05


Targets (difference [95% CI]; negative = system better):


,system,S1 vs B0,S2 vs B0,S5 vs H1b,S6 vs H1b,overall vs H1b,T1,T2',T3',ALL
0,GPV_K_v1,"+9.34 [+5.00, +14.48]","+3.34 [+0.44, +6.83]","-0.52 [-6.67, +6.37]","-10.60 [-42.03, +20.08]","+3.70 [-0.56, +10.16]",False,True,False,False
1,GPV_K-n[S],"+6.04 [+3.49, +8.88]","+0.60 [-1.12, +2.47]","-1.80 [-8.45, +5.32]","-18.88 [-42.75, +2.25]","-0.76 [-2.62, +1.11]",False,True,False,False
2,H1b-K3n[S],"+4.85 [+2.89, +6.96]","+0.94 [-0.80, +2.81]","-1.63 [-4.22, +0.63]","-10.28 [-28.15, +6.57]","-0.24 [-1.50, +1.16]",False,True,False,False
3,H1b[LM0],"+1.33 [-0.44, +3.11]","-1.60 [-3.34, +0.16]","+5.08 [-0.08, +11.52]","+24.45 [+12.60, +38.02]","+0.26 [-1.03, +1.64]",True,False,False,False
4,GPV_K-n[LM0],"+2.75 [+0.25, +5.49]","-2.11 [-3.86, -0.24]","+2.47 [-6.15, +12.07]","-1.20 [-28.62, +23.70]","-0.94 [-3.05, +1.25]",False,True,False,False
5,H1b-K3n[LM0],"+1.36 [-0.39, +3.10]","-1.50 [-3.26, +0.29]","+1.63 [-5.00, +8.97]","+11.57 [-9.08, +31.38]","-0.41 [-2.10, +1.39]",True,True,False,False
6,H1b[Z-S],"+10.74 [+8.11, +13.61]","+6.71 [+3.95, +9.59]","-4.88 [-8.53, -1.03]","-55.93 [-73.43, -40.28]","+1.08 [-0.52, +2.78]",False,True,False,False
7,GPV_K-n[Z-S],"+13.46 [+9.69, +17.69]","+6.48 [+3.79, +9.35]","-2.77 [-8.58, +4.18]","-66.03 [-88.98, -46.07]","+2.04 [-0.18, +4.39]",False,True,False,False
8,H1b-K3n[Z-S],"+10.51 [+7.95, +13.29]","+6.27 [+3.65, +8.99]","-5.17 [-9.35, -0.88]","-60.10 [-80.20, -40.25]","+1.35 [-0.56, +3.37]",False,True,False,False
9,H1b[Z-LM0],"+4.97 [+2.62, +7.50]","+1.21 [-1.18, +3.80]","-5.02 [-10.83, +2.20]","-32.32 [-51.35, -12.13]","-1.37 [-3.03, +0.38]",False,True,False,False


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3MN_Marigold_Normals/section4_bases
